# 10번. Train,Test셋 나누기

In [3]:
import pandas as pd

df = pd.read_parquet(r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번.파일(최종)(진)\M19_도매_소매업(최종)(진).parquet")

train = df[df["회계년도"].between(2012, 2021)].copy()
test  = df[df["회계년도"].between(2022, 2024)].copy()

print("Train shape:", train.shape)
print("Test  shape:", test.shape)

Train shape: (28111, 273)
Test  shape: (11797, 273)


행(row) = 연도 1개

→ train: 2012~2022 = 11행, test: 2023~2024 = 2행 

## 칼럼별 분포도 보기

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib
# import numpy as np
# from matplotlib.backends.backend_pdf import PdfPages

# matplotlib.rcParams['font.family'] = 'Malgun Gothic'  # Windows 한글 폰트
# matplotlib.rcParams['axes.unicode_minus'] = False

# df = pd.read_parquet(r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번.파일(최종)(진)\M19_도매_소매업(최종)(진).parquet")

# train = df[df["회계년도"].between(2012, 2021)].copy()

# # 숫자형 컬럼만 (식별 컬럼 제외)
# exclude = [
#     '사업자등록번호', '회계년도', '회사명', '회사명_norm', 'M코드',
#     '통계청 한국표준산업분류 코드 11차(대분류)',
#     '통계청 한국표준산업분류 11차(중분류)',
#     '부실라벨_ICR3년', '빅4감사',
# ]
# num_cols = [c for c in train.select_dtypes(include='number').columns if c not in exclude]

# print(f"히스토그램 그릴 컬럼 수: {len(num_cols)}")

# # PDF로 저장 (컬럼 수가 많아서 한 파일에 모아서 보기 편함)
# output_path = r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\칼럼별히스토그램\M19_도매_소매업_히스토그램_train.pdf"

# cols_per_page = 6  # 페이지당 2행 × 3열
# n_pages = int(np.ceil(len(num_cols) / cols_per_page))

# with PdfPages(output_path) as pdf:
#     for page in range(n_pages):
#         batch = num_cols[page * cols_per_page : (page + 1) * cols_per_page]
#         fig, axes = plt.subplots(2, 3, figsize=(18, 10))
#         axes = axes.flatten()

#         for i, col in enumerate(batch):
#             ax = axes[i]
#             data = train[col].dropna()

#             # 이상치 제외한 범위 (1~99 percentile) 로 x축 설정
#             p01, p99 = data.quantile(0.01), data.quantile(0.99)

#             ax.hist(data, bins=80, color='steelblue', edgecolor='none', alpha=0.8)
#             ax.axvline(p01, color='orange', linestyle='--', linewidth=1, label=f'p01: {p01:.2f}')
#             ax.axvline(p99, color='red',    linestyle='--', linewidth=1, label=f'p99: {p99:.2f}')
#             ax.axvline(data.median(), color='green', linestyle='-', linewidth=1, label=f'med: {data.median():.2f}')

#             ax.set_xlim(data.quantile(0.001), data.quantile(0.999))  # 극단치 제외하고 표시
#             ax.set_title(col, fontsize=11)
#             ax.legend(fontsize=7)
#             ax.set_xlabel('')
#             ax.tick_params(labelsize=8)

#             # 통계 텍스트
#             stats = (f"n={len(data):,}  NaN={train[col].isnull().sum():,}\n"
#                      f"min={data.min():.2f}  max={data.max():.2f}\n"
#                      f"mean={data.mean():.2f}  std={data.std():.2f}")
#             ax.text(0.98, 0.97, stats, transform=ax.transAxes,
#                     fontsize=7, va='top', ha='right',
#                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

#         # 빈 subplot 숨기기
#         for j in range(len(batch), len(axes)):
#             axes[j].set_visible(False)

#         fig.suptitle(f'Train 히스토그램 (page {page+1}/{n_pages})', fontsize=14)
#         plt.tight_layout()
#         pdf.savefig(fig)
#         plt.close(fig)

# print(f"저장 완료: {output_path}")

히스토그램 그릴 컬럼 수: 264
저장 완료: C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\칼럼별히스토그램\M19_도매_소매업_히스토그램_train.pdf


# 11번. 이상치 탐지 및 처리


이상치 탐지는 IQR(Interquartile Range) 방식을 사용하였다. 구체적으로, Train 데이터(2012~2022)를 기준으로 각 변수의 1사분위수(Q1)와 3사분위수(Q3)를 계산한 후 IQR(Q3-Q1)을 산출하였다. 이후 Q1 - 1.5 × IQR 미만, Q3 + 1.5 × IQR 초과하는 값을 이상치로 판단하였다.

탐지된 이상치는 윈저라이징(Winsorizing) 방식으로 처리하였다. 윈저라이징은 이상치를 제거하는 대신 각 경계값으로 대체하는 방법으로, 데이터 손실 없이 극단값의 영향을 완화할 수 있다. 또한 데이터 누수(Data Leakage) 방지를 위해 Train 데이터에서 산출한 경계값을 Test 데이터(2023~2024)에도 동일하게 적용하였다.

In [7]:
import pandas as pd
import numpy as np

df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\8,9번.파일(최종)(진)\M19_도매_소매업(최종)(진).parquet')

base_exclude = [
    '사업자등록번호', '회계년도', '회사명', '종업원',
    '통계청 한국표준산업분류 코드 11차(대분류)', '통계청 한국표준산업분류 11차(중분류)',
    '부실라벨_ICR3년', 'M코드', '회사명_norm', '업력', '빅4감사'
]

abs_amount_bases = [
    '자산총계(요약)(백만원)',
    '투자활동으로 인한 현금흐름(요약)(백만원)',
    '재무활동으로 인한 현금흐름(요약)(백만원)',
    'FCF', 'FCF_총자산', '유보율'
]

abs_amount_exclude = []
for base in abs_amount_bases:
    for suffix in ['', '_diff', '_ratio']:
        abs_amount_exclude.append(base + suffix)

all_exclude = set(base_exclude + abs_amount_exclude)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
target_cols = [c for c in numeric_cols if c not in all_exclude]

# train/test 분리
train = df[df['회계년도'].between(2012, 2021)].copy()
test  = df[df['회계년도'].between(2022, 2024)].copy()

# train만 연도별 1% 윈저라이징
for year in sorted(train['회계년도'].unique()):
    idx = train['회계년도'] == year
    for col in target_cols:
        s = train.loc[idx, col].dropna()
        if len(s) < 10:
            continue
        q_lo = s.quantile(0.01)
        q_hi = s.quantile(0.99)
        train.loc[idx, col] = train.loc[idx, col].clip(lower=q_lo, upper=q_hi)

print("Train shape:", train.shape)
print("Test  shape:", test.shape)
print("윈저라이징 완료 (train만 적용)")

Train shape: (28111, 273)
Test  shape: (11797, 273)
윈저라이징 완료 (train만 적용)


# 12번. 스케일링

재무비율 데이터는 변수 간 단위가 어느 정도 통일되어 있으나, 비율 간에도 값의 범위가 크게 다르고(예: 부채비율 수백 % vs 총자산회전율 1~2회) 극단값이 빈번하게 나타나는 특성이 있다. 이에 이상치에 강건한 Robust Scaler를 적용하였다. Robust Scaler는 평균과 표준편차 대신 중앙값(Median)과 IQR을 기준으로 데이터를 변환하기 때문에 극단값의 영향을 최소화할 수 있으며, 앞서 IQR 기반의 윈저라이징을 적용한 것과 동일한 기준을 사용한다는 점에서 일관성이 있다.

스케일링은 데이터 누수(Data Leakage) 방지를 위해 Train 데이터(2012~2022)에 대해서만 fit_transform을 수행하여 중앙값과 IQR을 학습하였으며, Test 데이터(2023~2024)에는 Train 기준으로 학습된 값을 그대로 적용하는 transform만 수행하였다.

In [12]:
from sklearn.preprocessing import MinMaxScaler

# 스케일링 제외 컬럼 (식별자, 레이블, 범주형만 제외)
scale_exclude = [
    '사업자등록번호', '회계년도', '회사명', '종업원',
    '통계청 한국표준산업분류 코드 11차(대분류)', '통계청 한국표준산업분류 11차(중분류)',
    '부실라벨_ICR3년', 'M코드', '회사명_norm', '업력', '빅4감사'
]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in scale_exclude]

# Min-Max 스케일링
scaler = MinMaxScaler()

# train: fit + transform
train[scale_cols] = scaler.fit_transform(train[scale_cols])

# test: train 기준 transform만 적용
test[scale_cols] = scaler.transform(test[scale_cols])

# 저장
train.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\train데이터\M19_도매_소매업_train.parquet', index=False)
test.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\test데이터\M19_도매_소매업_test.parquet', index=False)

print("Train shape:", train.shape)
print("Test  shape:", test.shape)
print(f"스케일링 적용 컬럼 수: {len(scale_cols)}")
print("저장 완료: M19_도매_소매업_train.parquet, M19_도매_소매업_test.parquet")

Train shape: (28111, 273)
Test  shape: (11797, 273)
스케일링 적용 컬럼 수: 262
저장 완료: M19_도매_소매업_train.parquet, M19_도매_소매업_test.parquet


이해 차원에서 적어둠
- fit : 데이터의 통계값(중앙값, IQR)을 학습하는 것
- transform : 학습한 통계값으로 데이터를 변환하는 것

# 저장

In [13]:
import pandas as pd

train = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\train데이터\M19_도매_소매업_train.parquet')
test  = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\test데이터\M19_도매_소매업_test.parquet')

print('=== Train ===')
print('shape:', train.shape)
print('연도:', sorted(train['회계년도'].unique()))

print()
print('=== Test ===')
print('shape:', test.shape)
print('연도:', sorted(test['회계년도'].unique()))

print()
print('Train == Test?', train.equals(test))

print()
print('=== dtype 분포 ===')
print(train.dtypes.value_counts())

print()
print('=== 수치 컬럼 기술통계 (앞 5개) ===')
float_cols = train.select_dtypes('float64').columns[:5].tolist()
print(train[float_cols].describe().round(4))

print()
print('=== NaN 많은 컬럼 Top 10 ===')
nan_ratio = train.isnull().mean().sort_values(ascending=False)
print(nan_ratio[nan_ratio > 0].head(10))

=== Train ===
shape: (28111, 273)
연도: [np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]

=== Test ===
shape: (11797, 273)
연도: [np.int64(2022), np.int64(2023), np.int64(2024)]

Train == Test? False

=== dtype 분포 ===
float64    263
str          5
int64        5
Name: count, dtype: int64

=== 수치 컬럼 기술통계 (앞 5개) ===
       자산총계(요약)(백만원)  투자활동으로 인한 현금흐름(요약)(백만원)  재무활동으로 인한 현금흐름(요약)(백만원)  \
count     28111.0000               28111.0000               28111.0000   
mean          0.0023                   0.6872                   0.2651   
std           0.0228                   0.0083                   0.0095   
min           0.0000                   0.0000                   0.0000   
25%           0.0002                   0.6875                   0.2649   
50%           0.0004                   0.6877                   0.2650   
75%           0.0007                   0.6878             